# Lesson 6A: Function Approximation - Theory

## Introduction

All lessons so far assume **tabular representation**: state and action values stored in tables (arrays/dicts).

This fails for large or continuous state spaces:
- Chess: 10^120 states (can't store table)
- Robotics: continuous joint angles (infinite states)
- Images: 256^(pixels) possible states

**Function approximation** replaces tables with parameterized functions:
$$V(s) \approx \hat{V}(s; \theta) = w^T \phi(s)$$

where φ(s) are hand-crafted or learned features, θ are learnable parameters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

np.random.seed(42)
print('Function Approximation in RL')

In [ ]:
# Visualize the curse
dimensions = np.arange(1, 11)
bins_per_dim = 10
num_states = bins_per_dim ** dimensions

plt.figure(figsize=(10, 5))
plt.semilogy(dimensions, num_states, 'o-', linewidth=2, markersize=8)
plt.axhline(y=1e6, color='r', linestyle='--', label='1M states (memory limit)')
plt.xlabel('Number of Dimensions')
plt.ylabel('Total States (log scale)')
plt.title(f'Curse of Dimensionality: {bins_per_dim} bins per dimension')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Specific examples
print("State space sizes for different dimensionalities (10 bins per dim):")
for d in [2, 5, 10, 20, 30]:
    states = 10 ** d
    print(f"  {d}D: 10^{d} = {states:.2e} states")

In [ ]:
# Example: approximating a value function on [0,1] with different features

def true_value(s):
    """True value function (target)"""
    return np.sin(4 * np.pi * s)

# Generate training samples
s_train = np.random.uniform(0, 1, 50)
v_train = true_value(s_train)

# Test range
s_test = np.linspace(0, 1, 200)
v_true = true_value(s_test)

# Method 1: Polynomial features
def poly_features(s, degree=3):
    return np.array([s**i for i in range(degree+1)])

# Method 2: RBF features
def rbf_features(s, centers=np.array([0.2, 0.5, 0.8]), sigma=0.1):
    return np.exp(-((s - centers)**2) / (2 * sigma**2))

# Method 3: Tile coding (binary features)
def tile_features(s, num_tiles=5):
    tile_size = 1.0 / num_tiles
    tiles = np.zeros(num_tiles)
    for i in range(num_tiles):
        if i * tile_size <= s < (i+1) * tile_size:
            tiles[i] = 1.0
    return tiles

# Fit each method via linear regression
from numpy.linalg import lstsq

# Polynomial
X_poly = np.array([poly_features(s) for s in s_train])
w_poly, _, _, _ = lstsq(X_poly, v_train, rcond=None)
v_poly = np.array([np.dot(w_poly, poly_features(s)) for s in s_test])

# RBF
X_rbf = np.array([rbf_features(s) for s in s_train])
w_rbf, _, _, _ = lstsq(X_rbf, v_train, rcond=None)
v_rbf = np.array([np.dot(w_rbf, rbf_features(s)) for s in s_test])

# Tile coding
X_tile = np.array([tile_features(s) for s in s_train])
w_tile, _, _, _ = lstsq(X_tile, v_train, rcond=None)
v_tile = np.array([np.dot(w_tile, tile_features(s)) for s in s_test])

# Visualize
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(s_test, v_true, 'k-', linewidth=3, label='True V(s)', zorder=5)
ax.scatter(s_train, v_train, alpha=0.5, s=30, label='Training samples', zorder=4)
ax.plot(s_test, v_poly, '--', linewidth=2, label='Polynomial (deg=3)', alpha=0.7)
ax.plot(s_test, v_rbf, '-.', linewidth=2, label='RBF (σ=0.1)', alpha=0.7)
ax.plot(s_test, v_tile, ':', linewidth=2, label='Tile coding (5 tiles)', alpha=0.7)
ax.set_xlabel('State s')
ax.set_ylabel('Value V(s)')
ax.set_title('Feature Representations: Polynomial vs RBF vs Tile Coding')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate errors
errors = {
    'Polynomial': np.mean((v_poly - v_true)**2),
    'RBF': np.mean((v_rbf - v_true)**2),
    'Tile': np.mean((v_tile - v_true)**2)
}
print("Mean squared errors on test set:")
for method, error in errors.items():
    print(f"  {method}: {error:.4f}")

## Feature Representations in Action

### Semi-Gradient Methods (Derivation)

Objective: minimize MSE of value predictions
$$J(w) = \mathbb{E}[(R + \gamma V(S'; w) - V(S; w))²]$$

Gradient descent update (standard):
$$w \leftarrow w - \alpha \frac{\partial J}{\partial w} = w - \alpha \mathbb{E}[(R + \gamma V(S'; w) - V(S; w)) \nabla_w (R + \gamma V(S'; w) - V(S; w))]$$

**Semi-gradient approximation**: Ignore gradient through the target (bootstrapped value).
$$w \leftarrow w - \alpha \mathbb{E}[(R + \gamma V(S'; w) - V(S; w)) \nabla_w V(S; w)]$$

This gives:
$$w \leftarrow w + \alpha (R + \gamma V(S'; w) - V(S; w)) \phi(S)$$

where $\phi(S) = \nabla_w V(S; w)$ are the features.

**Why "semi"?** We treat the target as if it doesn't depend on w, even though it does. This bias helps stability.

## The Curse of Dimensionality

### Why Tabular Fails

Suppose state = (x₁, x₂, ..., x_d) with each dimension discretized into k bins.

**Total states**: k^d

Examples:
- 2D with 10 bins each: 10² = 100 states (manageable)
- 5D with 10 bins each: 10⁵ = 100k states (getting large)
- 10D with 10 bins each: 10^10 = 10 billion states (impossible)
- Image: 84×84 pixels with 256 colors = 256^7056 states (absurd)

Storage required: O(k^d) — **exponential in dimension**.

### The Real Cost

Not just memory. Each state needs independent samples to learn its value. With k^d states, exploration becomes infeasible—we can't visit them all.

## Setup

## Why Function Approximation

### Generalization

Update value for one state → improves estimates for similar states (automatically via feature sharing)

### Scalability

Neural network: millions of weights model billions of states

### Continuous States

Linear combination of features naturally handles continuity

## Linear Function Approximation

### Form

$$\hat{V}(s; w) = w^T \phi(s) = \sum_i w_i \phi_i(s)$$

Features φ(s) ∈ R^d are fixed (hand-crafted). Learn weights w ∈ R^d via gradient descent.

### Example Features

- **Polynomial**: φ(s) = [1, s, s², s³, ...]
- **RBF**: φ(s) = [exp(-||s-c₁||²/σ²), ...] (Gaussian basis functions)
- **Tile coding**: Binary features marking membership in regions

### Gradient Descent Update

$$w \leftarrow w + \alpha (R + \gamma \hat{V}(S'; w) - \hat{V}(S; w)) \phi(S)$$

Natural extension of tabular TD: multiply TD error by features.

## Neural Network Approximation

### Deep Learning

Network learns **both** features and weights end-to-end:
$$\hat{V}(s; \theta) = \text{NN}(s; \theta)$$

where θ includes all layer weights and biases.

### Advantage

Automatic feature learning—network discovers relevant patterns.

### Challenge

Non-convex optimization—many local minima, harder to analyze convergence.

## Convergence and Challenges

### Issue 1: Divergence with Off-Policy

Tabular Q-learning converges. With function approximation and off-policy learning (Q-learning), can diverge.

**Fix**: Experience replay + target networks (see DQN, Lesson 7)

### Issue 2: Deadly Triad

Three ingredients together cause instability:
1. Function approximation
2. Bootstrapping (using V estimates as targets)
3. Off-policy learning

**Fix**: Reduce one (on-policy learning) or use stabilization tricks.

### Issue 3: Non-Stationary Target

Target is $R + \gamma V(S'; w)$ but w changes every step → moving target makes learning hard.

**Fix**: Target network (copy of V network, updated periodically)

## Key Insights

1. **Features matter**: Hand-crafted or learned via network
2. **Generalization**: Similar states share value estimates
3. **Scalability**: Handle large/continuous state spaces
4. **Convergence**: More complex than tabular (deadly triad)
5. **Tricks needed**: Experience replay, target networks, clipping

## Next: Lesson 6B (Practical)

Implement linear and neural approximators on CartPole—see scaling in action.